# 4.2 针对硬件的代码生成

通过编译器将高层计算图转换为目标硬件的高效代码。不同硬件架构有不同的计算特性和内存层次，需要针对性的代码生成策略。

## 什么是代码生成？

代码生成是编译器将高层的计算图或中间表示（IR）转换为目标硬件可直接执行的低级代码的过程。核心技术包括：
- **循环展开（Loop Unrolling）**：减少循环控制开销，增加指令级并行度
- **循环分块（Tiling）**：将大矩阵运算分解为适配缓存层次的小块
- **自动向量化**：利用SIMD指令同时处理多个数据元素
- **算子搜索（AutoTuning）**：搜索目标硬件上的最优实现配置

## 为什么用代码生成？

1. **硬件特异性**：同一算子在不同硬件上的最优实现完全不同
2. **内存层次利用**：针对缓存大小分块，使数据复用最大化
3. **编译期全局优化**：运行时框架只能调用预编译算子，代码生成可跨算子优化

## 代码生成的数学原理

**循环分块**：矩阵乘法$C = AB$，分块后$C_{m \times n} = \sum_{k} A_{m \times t_k} \cdot B_{t_k \times n}$，分块大小$(t_m, t_n, t_k)$满足：
$$t_m \cdot t_k \leq |L1|, \quad t_k \cdot t_n \leq |L1|, \quad t_m \cdot t_n \leq |RF|$$

**Roofline模型**：算术强度$I = \frac{\text{FLOPs}}{\text{Bytes}}$决定性能上界，代码生成的目标是使$I$超过阈值进入compute-bound区域。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

def benchmark(fn, *args, warmup=10, runs=100, **kwargs):
    """通用benchmark函数: warmup + 多次运行取平均"""
    for _ in range(warmup):
        fn(*args, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(runs):
        fn(*args, **kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / runs * 1000
    return elapsed

### 循环分块（Tiling）与内存层次

#### 什么是循环分块？

将大矩阵运算分解为适配缓存层次的小块，使每个小块的数据能完全放入L1/L2缓存，最大化数据复用。

#### 为什么循环分块有效？

1. **缓存友好**：分块后每个小块的数据在缓存中被多次复用，减少主存访问
2. **寄存器复用**：小块可放入寄存器，消除L1缓存访问延迟
3. **分块大小选择**：需匹配硬件缓存层次，过大则溢出缓存，过小则并行度不足

#### 分块的数学原理

矩阵乘法$C_{M \times N} = A_{M \times K} \cdot B_{K \times N}$，分块后：
$$C_{m \times n} = \sum_{k} A_{m \times t_k} \cdot B_{t_k \times n}$$

分块约束：
- $t_m \cdot t_k \leq |L1|$：A的分块放入L1
- $t_k \cdot t_n \leq |L1|$：B的分块放入L1
- $t_m \cdot t_n \leq |RF|$：C的分块放入寄存器文件

算术强度：$I = \frac{2 \cdot t_m \cdot t_n \cdot t_k}{4(t_m \cdot t_k + t_k \cdot t_n + t_m \cdot t_n)}$

In [ ]:
def analyze_tiling(M: int, N: int, K: int, tile_m: int, tile_n: int, tile_k: int,
                   l1_size_bytes: int = 32768, rf_size_bytes: int = 256):
    """分析分块配置的缓存适配性和算术强度"""
    a_block_bytes = tile_m * tile_k * 4
    b_block_bytes = tile_k * tile_n * 4
    c_block_bytes = tile_m * tile_n * 4

    l1_fit = (a_block_bytes + b_block_bytes) <= l1_size_bytes
    rf_fit = c_block_bytes <= rf_size_bytes

    flops = 2 * tile_m * tile_n * tile_k
    bytes_accessed = 4 * (tile_m * tile_k + tile_k * tile_n + tile_m * tile_n)
    arithmetic_intensity = flops / bytes_accessed

    return {
        'tile': (tile_m, tile_n, tile_k),
        'l1_fit': l1_fit,
        'rf_fit': rf_fit,
        'a_block_KB': a_block_bytes / 1024,
        'b_block_KB': b_block_bytes / 1024,
        'arithmetic_intensity': arithmetic_intensity,
    }

M, N, K = 1024, 1024, 1024
tile_configs = [
    (16, 16, 16),
    (32, 32, 32),
    (64, 64, 64),
    (128, 128, 32),
    (64, 64, 128),
]

print(f"=== 循环分块分析 (矩阵: {M}x{K} @ {K}x{N}) ===")
print(f"L1缓存: 32KB, 寄存器文件: 256B")
print(f"\n{'分块(tm,tn,tk)':<20} {'L1适配':<8} {'RF适配':<8} {'A块(KB)':<10} {'B块(KB)':<10} {'算术强度':<10}")
print("-" * 66)
for tile in tile_configs:
    info = analyze_tiling(M, N, K, *tile)
    print(f"{str(info['tile']):<20} {'Y' if info['l1_fit'] else 'N':<8} {'Y' if info['rf_fit'] else 'N':<8} "
          f"{info['a_block_KB']:<10.1f} {info['b_block_KB']:<10.1f} {info['arithmetic_intensity']:<10.2f}")

print(f"\n分块选择原则:")
print(f"1. 确保A块+B块 ≤ L1缓存大小")
print(f"2. 确保C块 ≤ 寄存器文件大小")
print(f"3. 在满足1和2的前提下，最大化算术强度")
print(f"4. 实际最优分块需在目标硬件上实测 (AutoTuning)")

### TVM风格算子搜索

#### 什么是算子搜索？

TVM将算子实现参数化（如分块大小、展开因子、向量化宽度），构成调度空间（Schedule Space），然后在目标硬件上实测每个配置的性能，选择最优配置。

#### 为什么需要搜索？

1. **硬件行为难以建模**：缓存替换策略、预取机制等硬件行为难以精确建模
2. **参数交互复杂**：分块大小、展开因子之间存在复杂的交互效应
3. **硬件差异大**：不同硬件微架构的最优配置差异显著

#### 算子搜索的数学原理

设调度空间为$\mathcal{S} = \{s_1, s_2, \ldots, s_N\}$，每个调度$s_i$对应运行时延迟$T(s_i)$：
$$s^* = \arg\min_{s \in \mathcal{S}} T(s)$$

- **AutoTVM**：使用XGBoost成本模型预测$\hat{T}(s)$，减少实测次数
- **Ansor**：使用蒙特卡洛树搜索（MCTS）在更大的调度空间中搜索
- 搜索空间大小通常为$10^{10}$级别，需要高效的搜索策略

In [ ]:
class MatMulTuningSpace:
    """模拟TVM的矩阵乘法调优空间"""
    def __init__(self, M: int, N: int, K: int):
        self.M = M
        self.N = N
        self.K = K
        self.tile_sizes = [(16, 16), (32, 32), (64, 64), (128, 128)]
        self.unroll_factors = [1, 2, 4, 8]
        self.vector_lengths = [1, 4, 8]

    def enumerate_configs(self) -> list:
        configs = []
        for tile in self.tile_sizes:
            for unroll in self.unroll_factors:
                for vec in self.vector_lengths:
                    configs.append({
                        'tile_m': tile[0], 'tile_n': tile[1],
                        'unroll': unroll, 'vector_len': vec
                    })
        return configs

    def estimate_performance(self, config: dict) -> float:
        """模拟性能评估（实际TVM会在硬件上实测）"""
        tile_m = config['tile_m']
        tile_n = config['tile_n']
        unroll = config['unroll']
        vec = config['vector_len']

        cache_efficiency = min(1.0, (tile_m * tile_n) / (64 * 64))
        parallelism = min(1.0, unroll / 4)
        vector_efficiency = min(1.0, vec / 8)
        register_pressure = min(1.0, 1.0 / max(1, tile_m * tile_n / 1024))

        score = cache_efficiency * 0.4 + parallelism * 0.2 + vector_efficiency * 0.3 + register_pressure * 0.1
        return score

M, N, K = 512, 512, 512
tuner = MatMulTuningSpace(M, N, K)
configs = tuner.enumerate_configs()

print(f"=== TVM风格算子调优空间 ===")
print(f"矩阵乘法: {M}x{K} @ {K}x{N}")
print(f"候选配置数: {len(configs)}")

results = [(c, tuner.estimate_performance(c)) for c in configs]
results.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop-5配置:")
for config, score in results[:5]:
    print(f"  tile=({config['tile_m']},{config['tile_n']}) unroll={config['unroll']} vec={config['vector_len']} score={score:.4f}")

print(f"\nTVM调优流程:")
print(f"1. 定义算子计算逻辑")
print(f"2. 枚举可能的调度配置(tile/unroll/vectorize)")
print(f"3. 在目标硬件上实测每个配置")
print(f"4. 选择最优配置生成代码")
print(f"5. 缓存调优结果供后续部署使用")

### PyTorch 原生量化推理

#### 什么是量化推理？

将模型权重和/或激活值从FP32转换为INT8/INT4表示，利用硬件的低精度算术单元加速推理。

#### 为什么用量化推理？

1. **计算吞吐量提升**：INT8 Tensor Core吞吐量是FP32的2-4倍
2. **内存带宽减少4倍**：INT8每元素1字节 vs FP32每元素4字节
3. **缓存有效容量增加4倍**：可容纳更多数据

#### 量化推理的数学原理

均匀量化：$x_{\text{int}} = \text{clamp}(\lfloor x/s \rceil + z, q_{\min}, q_{\max})$
反量化：$x_{\text{fp32}} \approx s \cdot (x_{\text{int}} - z)$
其中$s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}$为缩放因子，$z$为零点。

INT8矩阵乘法：$Y_{\text{int32}} = X_{\text{int8}} \cdot W_{\text{int8}}$，然后$Y_{\text{fp32}} = s_x \cdot s_w \cdot Y_{\text{int32}} + b$

In [ ]:
class QuantizedLinear(nn.Module):
    """PyTorch原生动态量化Linear层
    使用torch.quantization实现真正的INT8推理"""
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.fp32_linear = nn.Linear(in_dim, out_dim)

    def quantize_dynamic(self) -> nn.Module:
        """动态量化: 仅权重INT8化，激活运行时量化"""
        return torch.quantization.quantize_dynamic(
            self.fp32_linear, {nn.Linear}, dtype=torch.qint8
        )

in_dim, out_dim = 256, 128
q_linear = QuantizedLinear(in_dim, out_dim)
x = torch.randn(64, in_dim)

with torch.no_grad():
    out_fp32 = q_linear.fp32_linear(x)

try:
    quantized = q_linear.quantize_dynamic()
    with torch.no_grad():
        out_int8 = quantized(x)
    diff = (out_fp32 - out_int8).norm() / out_fp32.norm() * 100

    fp32_time = benchmark(lambda: q_linear.fp32_linear(x))
    int8_time = benchmark(lambda: quantized(x))

    fp32_weight_mem = sum(p.numel() * p.element_size() for p in q_linear.fp32_linear.parameters())
    print(f"=== PyTorch原生动态量化 ===")
    print(f"FP32输出范数: {out_fp32.norm():.4f}")
    print(f"INT8输出范数: {out_int8.norm():.4f}")
    print(f"相对误差: {diff:.4f}%")
    print(f"FP32权重内存: {fp32_weight_mem/1024:.2f} KB")
    print(f"FP32时间: {fp32_time:.4f} ms")
    print(f"INT8时间: {int8_time:.4f} ms")
    print(f"加速比: {fp32_time/int8_time:.2f}x")
except Exception as e:
    print(f"动态量化不可用: {e}")
    print(f"使用模拟量化演示...")

    class SimulatedQuantizedLinear(nn.Module):
        """模拟INT8量化矩阵乘法（教学用）
        注意: 量化后仍转为float计算，不会产生实际INT8加速效果。
        真正的INT8加速需要硬件INT8支持 + 专用kernel + per-channel量化"""
        def __init__(self, fp32_linear: nn.Linear):
            super().__init__()
            self.weight = fp32_linear.weight.data
            self.bias = fp32_linear.bias.data

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            w_scale = self.weight.abs().max(dim=1, keepdim=True)[0] / 127.0
            w_int8 = torch.round(self.weight / w_scale.clamp(min=1e-8)).clamp(-128, 127)
            x_scale = x.abs().max(dim=-1, keepdim=True)[0] / 127.0
            x_int8 = torch.round(x / x_scale.clamp(min=1e-8)).clamp(-128, 127)
            result_int32 = x_int8.float() @ w_int8.float().T
            result_fp32 = result_int32 * (x_scale * w_scale.T)
            return result_fp32 + self.bias

    sim_q = SimulatedQuantizedLinear(q_linear.fp32_linear)
    with torch.no_grad():
        out_sim = sim_q(x)
    diff = (out_fp32 - out_sim).norm() / out_fp32.norm() * 100
    print(f"\n模拟量化相对误差: {diff:.4f}%")

print(f"\n产业界量化方案:")
print(f"1. PyTorch动态量化: torch.quantization.quantize_dynamic (最简单)")
print(f"2. PyTorch静态量化: 需校准数据集，精度更高")
print(f"3. ONNX Runtime量化: onnxruntime.quantization.quantize_static")
print(f"4. GPTQ/AWQ: LLM专用权重量化 (4bit)")
print(f"5. bitsandbytes: NF4量化 (QLoRA微调)")

### 编译框架对比

#### 什么是编译框架？

深度学习编译框架是将训练好的模型转换为目标硬件高效执行代码的工具链。

#### 为什么需要多种编译框架？

1. **硬件生态不同**：NVIDIA/Apple/ARM/RISC-V需要不同的代码生成后端
2. **部署场景不同**：云端推理/端侧推理对编译时间和运行性能的权衡不同
3. **模型类型不同**：CNN/Transformer/扩散模型的最优编译策略不同

#### 核心编译策略对比

| 策略 | 代表框架 | 原理 | 适用场景 |
|------|---------|------|----------|
| 搜索式编译 | TVM/Ansor | 在调度空间中搜索最优kernel配置 | 跨平台、新硬件适配 |
| 多级IR编译 | MLIR/XLA | 逐级lowering + 逐级优化 | TPU/自定义加速器 |
| 模板式编译 | TensorRT | 预定义优化模板 + 层融合 | NVIDIA GPU推理 |
| 追踪式编译 | torch.compile | 运行时追踪 + JIT编译 | PyTorch快速部署 |

In [ ]:
print(f"{'框架':<20} {'核心方法':<25} {'支持硬件':<25} {'特点':<20}")
print("-" * 90)
frameworks = [
    ("TVM/Apache TVM", "AutoTVM/Ansor搜索", "CPU/GPU/NPU", "跨平台，搜索最优kernel"),
    ("MLIR/XLA", "多级IR逐级lowering", "CPU/GPU/TPU", "编译器基础设施"),
    ("TensorRT", "层融合+精度校准", "NVIDIA GPU", "GPU推理极致优化"),
    ("Core ML Tools", "图优化+ANE编译", "Apple Silicon", "Apple生态最优"),
    ("ONNX Runtime", "图优化+量化+EP", "CPU/GPU/NPU", "通用推理引擎"),
    ("torch.compile", "TorchDynamo+Inductor", "CPU/GPU", "PyTorch原生编译"),
    ("ExecuTorch", "AOTInductor+Delegate", "CPU/NPU/DSP", "Meta端侧推理框架"),
]
for name, method, hw, feature in frameworks:
    print(f"{name:<20} {method:<25} {hw:<25} {feature:<20}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. NVIDIA GPU: TensorRT (最成熟，性能最优)")
print(f"2. Apple端侧: Core ML (ANE加速，生态集成)")
print(f"3. 通用CPU/NPU: ONNX Runtime 或 TVM")
print(f"4. PyTorch快速验证: torch.compile (开发效率高)")
print(f"5. 移动端部署: ExecuTorch (PyTorch原生端侧方案)")
print(f"6. 新硬件适配: TVM/Ansor (自动搜索最优配置)")

### 产业级实战：ONNX 导出 + ONNX Runtime 推理

ONNX 是产业界最通用的模型交换格式。以下展示完整的端到端流程：
PyTorch模型 → ONNX导出 → ONNX Runtime推理 → 性能对比

#### 关键步骤

1. **模型导出**：`torch.onnx.export()` 将PyTorch模型转换为ONNX格式
2. **模型验证**：`onnx.checker.check_model()` 验证导出的ONNX模型是否合法
3. **推理对比**：比较PyTorch和ONNX Runtime的输出差异和推理速度

#### 导出注意事项

- 需要指定`opset_version`（推荐14+），不同opset支持的算子不同
- 动态形状需通过`dynamic_axes`参数指定
- 导出等价性：$\|f_{\text{PyTorch}}(x) - f_{\text{ONNX}}(x)\|_\infty < 10^{-5}$

In [ ]:
import io

class SmallGPT(nn.Module):
    """轻量GPT模型（不依赖transformers库）"""
    def __init__(self, vocab_size: int = 1000, dim: int = 128, depth: int = 4, n_heads: int = 4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                'attn': nn.MultiheadAttention(dim, n_heads, batch_first=True),
                'ffn': nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim)),
                'ln1': nn.LayerNorm(dim),
                'ln2': nn.LayerNorm(dim),
            }) for _ in range(depth)
        ])
        self.ln_f = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.embed(input_ids)
        for block in self.blocks:
            h = block['ln1'](x)
            h, _ = block['attn'](h, h, h)
            x = x + h
            x = x + block['ffn'](block['ln2'](x))
        x = self.ln_f(x)
        return self.head(x)

gpt_model = SmallGPT(vocab_size=1000, dim=128, depth=4, n_heads=4)
gpt_model.eval()
dummy_ids = torch.randint(0, 1000, (1, 32))

HAS_ORT = False
onnx_size = 0
pt_output = None

try:
    import onnx
    import onnxruntime as ort
    HAS_ORT = True
except ImportError:
    print(f"onnx/onnxruntime未安装，跳过ORT推理")

try:
    onnx_buffer = io.BytesIO()
    torch.onnx.export(
        gpt_model,
        dummy_ids,
        onnx_buffer,
        input_names=['input_ids'],
        output_names=['logits'],
        dynamic_axes={'input_ids': {0: 'batch', 1: 'seq_len'}, 'logits': {0: 'batch', 1: 'seq_len'}},
        opset_version=14,
        do_constant_folding=True,
    )
    onnx_buffer.seek(0)
    onnx_size = len(onnx_buffer.getvalue())

    with torch.no_grad():
        pt_output = gpt_model(dummy_ids).numpy()

    print(f"=== ONNX导出成功 ===")
    print(f"模型: SmallGPT (4层, 128维, 1000词表)")
    print(f"参数量: {sum(p.numel() for p in gpt_model.parameters()):,}")
    print(f"ONNX文件大小: {onnx_size/1024:.1f} KB")
    print(f"PyTorch输出shape: {pt_output.shape}")

    if HAS_ORT:
        onnx_model = onnx.load(onnx_buffer)
        onnx.checker.check_model(onnx_model)
        sess = ort.InferenceSession(onnx_buffer.getvalue())
        ort_output = sess.run(None, {'input_ids': dummy_ids.numpy()})[0]
        max_diff = np.abs(pt_output - ort_output).max()
        print(f"ONNX Runtime输出shape: {ort_output.shape}")
        print(f"最大数值差异: {max_diff:.8f}")
        print(f"数值一致性: {'PASS' if max_diff < 1e-5 else 'FAIL'}")
    else:
        print(f"\n安装ONNX Runtime: pip install onnx onnxruntime")
except Exception as e:
    print(f"ONNX导出失败: {e}")
    print(f"常见原因: 缺少onnxscript依赖 (pip install onnxscript)")
    print(f"或使用旧版API: torch.onnx.export(..., dynamo=False)" if hasattr(torch.onnx, 'export') else "")

In [ ]:
if HAS_ORT:
    try:
        sess
        HAS_SESS = True
    except NameError:
        HAS_SESS = False
else:
    HAS_SESS = False

if HAS_SESS:
    print(f"=== 性能对比: PyTorch vs ONNX Runtime ===")

    def benchmark_torch(model, input_ids, warmup=10, runs=50):
        with torch.no_grad():
            for _ in range(warmup):
                model(input_ids)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            start = time.perf_counter()
            for _ in range(runs):
                model(input_ids)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            return (time.perf_counter() - start) / runs * 1000

    def benchmark_ort(session, input_ids_np, warmup=10, runs=50):
        for _ in range(warmup):
            session.run(None, {'input_ids': input_ids_np})
        start = time.perf_counter()
        for _ in range(runs):
            session.run(None, {'input_ids': input_ids_np})
        return (time.perf_counter() - start) / runs * 1000

    batch_sizes = [1, 4, 8]
    seq_len = 32

    print(f"{'Batch':<8} {'PyTorch(ms)':<14} {'ORT(ms)':<14} {'加速比':<10}")
    print("-" * 46)
    for bs in batch_sizes:
        ids = torch.randint(0, 1000, (bs, seq_len))
        pt_time = benchmark_torch(gpt_model, ids)
        ort_time = benchmark_ort(sess, ids.numpy())
        speedup = pt_time / ort_time
        print(f"{bs:<8} {pt_time:<14.2f} {ort_time:<14.2f} {speedup:<10.2f}x")

    print(f"\nONNX Runtime优化选项:")
    sess_opts = ort.SessionOptions()
    sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess_opts.intra_op_num_threads = 4
    sess_optimized = ort.InferenceSession(onnx_buffer.getvalue(), sess_opts)

    ids_b1 = torch.randint(0, 1000, (1, seq_len))
    ort_default = benchmark_ort(sess, ids_b1.numpy())
    ort_opt = benchmark_ort(sess_optimized, ids_b1.numpy())
    print(f"  默认ORT: {ort_default:.2f} ms")
    print(f"  优化ORT: {ort_opt:.2f} ms")

    print(f"\n产业界ONNX部署最佳实践:")
    print(f"1. 导出: torch.onnx.export + do_constant_folding=True + opset_version=14+")
    print(f"2. 验证: onnx.checker.check_model + 数值一致性对比")
    print(f"3. 优化: ORT_ENABLE_ALL + 线程数调优 + IO Binding")
    print(f"4. 量化: onnxruntime.quantization.quantize_static (INT8)")
    print(f"5. 常见问题: 动态shape → dynamic_axes; 自定义算子 → onnx-script注册")
else:
    print(f"=== ONNX Runtime未安装，跳过性能对比 ===")
    print(f"安装命令: pip install onnx onnxruntime")
    print(f"\n产业界ONNX部署最佳实践:")
    print(f"1. 导出: torch.onnx.export + do_constant_folding=True + opset_version=14+")
    print(f"2. 验证: onnx.checker.check_model + 数值一致性对比")
    print(f"3. 优化: ORT_ENABLE_ALL + 线程数调优 + IO Binding")
    print(f"4. 量化: onnxruntime.quantization.quantize_static (INT8)")
    print(f"5. 常见问题: 动态shape → dynamic_axes; 自定义算子 → onnx-script注册")

### Triton Kernel 编写：GPU代码生成的实践

#### 什么是Triton？

Triton是OpenAI开发的GPU编程语言，介于CUDA和高级框架之间。TorchInductor后端自动将FX Graph编译为Triton代码，是PyTorch 2.0编译优化的核心。

#### 为什么学习Triton？

1. **理解编译器输出**：torch.compile生成的就是Triton代码，理解Triton有助于调试和优化
2. **自定义算子**：当框架内置算子不够用时，Triton比CUDA更易编写
3. **产业界趋势**：Triton正在成为GPU编程的事实标准（PyTorch/JAX/XLA均支持）

#### Triton vs CUDA

| 特性 | CUDA | Triton |
|------|------|--------|
| 编程模型 | 手动管理线程 | 基于block的编程 |
| 内存管理 | 手动shared memory | 自动缓存管理 |
| 开发效率 | 低 | 高 |
| 性能上限 | 最高 | 接近CUDA |
| 学习曲线 | 陡峭 | 平缓 |

In [ ]:
try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
except ImportError:
    HAS_TRITON = False
    print(f"Triton未安装，使用PyTorch模拟演示")

if HAS_TRITON:
    @triton.jit
    def fused_linear_relu_kernel(
        x_ptr, w_ptr, b_ptr, out_ptr,
        N, K,
        BLOCK_SIZE: tl.constexpr,
    ):
        row_idx = tl.program_id(0)
        row_start = row_idx * K
        x_ptrs = x_ptr + row_start + tl.arange(0, BLOCK_SIZE)
        mask = tl.arange(0, BLOCK_SIZE) < K
        x_row = tl.load(x_ptrs, mask=mask, other=0.0)
        for col in range(N):
            w_ptrs = w_ptr + col * K + tl.arange(0, BLOCK_SIZE)
            w_row = tl.load(w_ptrs, mask=mask, other=0.0)
            dot = tl.sum(x_row * w_row)
            bias = tl.load(b_ptr + col)
            val = tl.maximum(dot + bias, 0.0)
            tl.store(out_ptr + row_idx * N + col, val)

    print(f"Triton已安装，fused_linear_relu_kernel已定义")
    print(f"此kernel将Linear+ReLU融合为单个GPU kernel")
else:
    print(f"\n=== Triton Kernel 示例 (伪代码) ===")
    print(f"""
    @triton.jit
    def fused_linear_relu_kernel(
        x_ptr, w_ptr, b_ptr, out_ptr,
        N, K, BLOCK_SIZE: tl.constexpr,
    ):
        # 每个program处理一行输入
        row_idx = tl.program_id(0)
        # 加载输入行
        x_row = tl.load(x_ptr + row_idx * K + tl.arange(0, BLOCK_SIZE))
        # 逐列计算矩阵乘 + bias + ReLU
        for col in range(N):
            w_col = tl.load(w_ptr + col * K + tl.arange(0, BLOCK_SIZE))
            dot = tl.sum(x_row * w_col)
            bias = tl.load(b_ptr + col)
            val = tl.maximum(dot + bias, 0.0)  # ReLU融合
            tl.store(out_ptr + row_idx * N + col, val)
    """)

print(f"\n=== torch.compile生成的Triton代码查看方法 ===")
print(f"import os")
print(f"os.environ['TORCH_LOGS'] = '+inductor'  # 查看Inductor生成的Triton代码")
print(f"os.environ['TORCH_LOGS'] = '+output_code'  # 查看最终生成的代码")
print(f"\n产业界Triton使用:")
print(f"1. torch.compile后端: 自动生成Triton kernel")
print(f"2. 自定义算子: Flash Attention v2使用Triton编写")
print(f"3. 性能调优: 修改BLOCK_SIZE等参数优化kernel")
print(f"4. 安装: pip install triton (需要CUDA GPU)")

### AutoTVM vs AutoScheduler：搜索策略的演进

#### AutoTVM（基于模板的搜索）

需要人工编写调度模板（schedule template），定义搜索空间，然后使用XGBoost成本模型在空间中搜索最优配置。

#### AutoScheduler / Ansor（无模板搜索）

无需人工模板，使用蒙特卡洛树搜索（MCTS）自动生成调度规则，搜索空间更大但更通用。

#### 对比

| 特性 | AutoTVM | AutoScheduler/Ansor |
|------|---------|---------------------|
| 搜索空间 | 人工定义模板 | 自动生成 |
| 搜索策略 | XGBoost成本模型 | MCTS + 进化搜索 |
| 搜索空间大小 | ~10^8 | ~10^10 |
| 人工介入 | 需要编写模板 | 无需 |
| 搜索时间 | 数小时 | 数十小时 |
| 新硬件适配 | 需重写模板 | 自动适配 |
| 最优性能 | 模板好则最优 | 通常更优 |

#### 编译缓存

端侧部署时编译时间很长（数小时），编译缓存机制至关重要：
- 首次编译：搜索最优配置 → 生成代码 → 缓存结果
- 后续部署：直接加载缓存 → 跳过搜索 → 即时部署
- TVM使用`.log`文件缓存AutoTVM结果，Ansor使用数据库缓存

In [ ]:
class CompileCacheSimulator:
    """模拟编译缓存机制"""
    def __init__(self):
        self.cache = {}
        self.compile_count = 0
        self.cache_hit_count = 0

    def get_or_compile(self, model_key: str, compile_time_ms: float = 5000.0) -> dict:
        if model_key in self.cache:
            self.cache_hit_count += 1
            return {'time_ms': 1.0, 'source': 'cache_hit'}
        else:
            self.compile_count += 1
            self.cache[model_key] = True
            return {'time_ms': compile_time_ms, 'source': 'compile'}

cache_sim = CompileCacheSimulator()

model_keys = [
    ('resnet18', 'cuda', 'fp32'),
    ('resnet18', 'cuda', 'int8'),
    ('mobilenetv2', 'cuda', 'fp32'),
    ('resnet18', 'cuda', 'fp32'),
    ('mobilenetv2', 'cuda', 'fp32'),
]

print(f"=== 编译缓存模拟 ===")
print(f"{'模型键':<30} {'耗时(ms)':<12} {'来源':<10}")
print("-" * 52)
total_compile_time = 0
for key in model_keys:
    key_str = f"{key[0]}_{key[1]}_{key[2]}"
    result = cache_sim.get_or_compile(key_str)
    total_compile_time += result['time_ms']
    print(f"{key_str:<30} {result['time_ms']:<12.1f} {result['source']:<10}")

print(f"\n编译次数: {cache_sim.compile_count}")
print(f"缓存命中: {cache_sim.cache_hit_count}")
print(f"总耗时: {total_compile_time:.1f} ms")
print(f"缓存节省: {(1 - total_compile_time / (len(model_keys) * 5000.0)) * 100:.0f}%")

print(f"\n产业界编译缓存实践:")
print(f"1. TVM: .log文件缓存AutoTVM调优结果")
print(f"2. torch.compile: __pycache__缓存编译结果")
print(f"3. TensorRT: .engine文件缓存序列化引擎")
print(f"4. ONNX Runtime: ORT模型缓存 + 预编译EP")
print(f"5. CI/CD: 编译结果作为artifact缓存，部署时直接下载")

### 产业级实战：AOTInductor 编译为C++共享库

#### 什么是AOTInductor？

PyTorch 2.1+ 提供的Ahead-Of-Time编译器，将模型编译为独立的C++共享库（.so/.dll），无需Python运行时即可推理。

#### 为什么用AOTInductor？

1. **无Python依赖**：编译后的.so文件可在纯C++环境中运行
2. **启动快**：跳过Python解释器和JIT编译，直接加载共享库
3. **端侧友好**：适合嵌入式设备、移动端等无Python环境

#### AOTInductor编译流程

```bash
# 编译命令
python -m torch._inductor.aot_compile \
    --model-class MyModel \
    --model-file model.py \
    --output-dir ./aot_output \
    --dynamic-shapes

# 输出: model.so (C++共享库) + model.json (元数据)
```

#### C++推理代码

```cpp
#include <torch/script.h>
auto model = torch::inductor::AOTInductorModel::create(
    "model.so", "model.json");
auto output = model->forward({input_tensor});
```

In [ ]:
class AOTInductorDemo(nn.Module):
    """AOTInductor编译示例模型"""
    def __init__(self, dim: int = 128):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 4)
        self.fc2 = nn.Linear(dim * 4, dim)
        self.ln = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.fc1(x)
        h = F.gelu(h)
        h = self.fc2(h)
        return self.ln(h + x)

aot_model = AOTInductorDemo(dim=128)
aot_model.eval()
dummy_x = torch.randn(4, 32, 128)

try:
    from torch._inductor import aot_compile
    HAS_AOT = True
except ImportError:
    HAS_AOT = False

print(f"=== AOTInductor 编译演示 ===")
print(f"模型: AOTInductorDemo (128维, 2层MLP + LayerNorm + Residual)")
print(f"参数量: {sum(p.numel() for p in aot_model.parameters()):,}")

if HAS_AOT:
    try:
        import tempfile
        import os
        with tempfile.TemporaryDirectory() as tmpdir:
            so_path = aot_compile(
                aot_model,
                [(dummy_x,)],
            )
            print(f"编译成功: {so_path}")
            print(f"共享库大小: {os.path.getsize(so_path)/1024:.1f} KB")
    except Exception as e:
        print(f"AOT编译失败: {e}")
        print(f"常见原因: 需要PyTorch 2.1+ 且支持当前平台")
else:
    print(f"AOTInductor不可用 (需要PyTorch 2.1+)")

eager_time = benchmark(lambda: aot_model(dummy_x))
print(f"\nEager模式延迟: {eager_time:.4f} ms")

try:
    compiled_aot = torch.compile(aot_model, mode='reduce-overhead')
    with torch.no_grad():
        for _ in range(5):
            compiled_aot(dummy_x)
    compiled_time = benchmark(lambda: compiled_aot(dummy_x))
    print(f"torch.compile延迟: {compiled_time:.4f} ms (加速{eager_time/compiled_time:.2f}x)")
except Exception as e:
    print(f"torch.compile不可用: {e}")

print(f"\nAOTInductor vs torch.compile:")
print(f"1. torch.compile: JIT编译，运行时生成代码，需要Python环境")
print(f"2. AOTInductor: AOT编译，提前生成.so，无需Python环境")
print(f"3. 适用场景: AOTInductor适合端侧/嵌入式部署")
print(f"4. 编译命令: python -m torch._inductor.aot_compile")

### 本章总结

#### 代码生成技术全景

| 技术 | 原理 | 代表框架 | 适用场景 |
|------|------|---------|----------|
| 循环分块 | 适配缓存层次 | 所有编译器 | 通用优化 |
| 算子搜索 | 硬件实测最优配置 | TVM/Ansor | 新硬件适配 |
| 模板式编译 | 预定义优化模板 | TensorRT | NVIDIA GPU |
| JIT编译 | 运行时生成代码 | torch.compile | 快速验证 |
| AOT编译 | 提前编译为C++ | AOTInductor | 端侧部署 |
| 量化推理 | 低精度算术 | ONNX RT/ORT | 内存受限场景 |

#### 产业界最佳实践

1. **NVIDIA GPU推理**: TensorRT (层融合+INT8+kernel auto-tuning)
2. **通用CPU/NPU**: ONNX Runtime (图优化+量化+多EP)
3. **PyTorch快速验证**: torch.compile (一行代码加速)
4. **端侧部署**: AOTInductor → .so文件 → 纯C++推理
5. **新硬件适配**: TVM/Ansor (自动搜索最优配置)
6. **Apple端侧**: Core ML Tools (ANE加速)
7. **编译缓存**: 首次编译耗时，后续部署直接加载缓存